In [10]:
"""
Password Strength Checker - Modern Tkinter GUI (Crimson theme)

Required files (keep them in the same folder as this script):
    password_model.pkl
    tfidf_vectorizer.pkl

Run:
    python password_strength_ui.py

IMPORTANT:
This prediction pipeline assumes the model was trained with features in this order:
    [TF-IDF character features..., password length, lowercase frequency]

The input feature order and preprocessing MUST match the training pipeline.
"""

import tkinter as tk
from tkinter import messagebox
from pathlib import Path

import joblib
import numpy as np


# --------------------------- Load saved artifacts ---------------------------

try:
    BASE_DIR = Path(__file__).resolve().parent   # running as a .py script
except NameError:
    BASE_DIR = Path.cwd()                        # running in Jupyter / interactive

MODEL_PATH = BASE_DIR / "password_model.pkl"
VECTORIZER_PATH = BASE_DIR / "tfidf_vectorizer.pkl"


def load_artifacts():
    missing = [str(path.name) for path in (MODEL_PATH, VECTORIZER_PATH) if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "Missing required file(s): " + ", ".join(missing) +
            "\nPlace the .pkl files in the same folder as this Python script."
        )

    model = joblib.load(MODEL_PATH)
    vectorizer = joblib.load(VECTORIZER_PATH)
    return model, vectorizer


# ------------------------------ Prediction ---------------------------------

def get_password_features(password, vectorizer, model):
    """Build features using the same order as the notebook's prediction code."""
    if not password:
        raise ValueError("Please enter a password.")

    # Use the already-fitted vectorizer. Do not fit it again on user input.
    tfidf_features = vectorizer.transform([password]).toarray()

    password_length = len(password)
    lowercase_frequency = sum(char.islower() for char in password) / password_length

    # Notebook pipeline: TF-IDF features + length + lowercase frequency.
    features = np.hstack((
        tfidf_features,
        np.array([[password_length, lowercase_frequency]], dtype=float)
    ))

    expected = getattr(model, "n_features_in_", None)
    if expected is not None and features.shape[1] != expected:
        raise ValueError(
            f"Feature mismatch: this app created {features.shape[1]} features, "
            f"but the saved model expects {expected}.\n\n"
            "The training and prediction pipelines must use exactly the same "
            "features in the same order. Check whether you included additional "
            "features (uppercase, digit, or special-character frequency) when "
            "training the model."
        )

    return features


# Strength colours are deliberately separate from the red brand accent so that
# "weak" never blends into the theme.
WEAK_C, NORMAL_C, STRONG_C, UNKNOWN_C = "#FF5A5F", "#FFB020", "#2DD4A7", "#A1A1AA"


def predict_strength(password):
    """Returns (label, color, description, meter_level 0-3)."""
    features = get_password_features(password, vectorizer, model)
    prediction = model.predict(features)[0]

    labels = {
        0: ("WEAK", WEAK_C, "This password may be easy to guess.", 1),
        1: ("NORMAL", NORMAL_C, "Consider making it longer and less predictable.", 2),
        2: ("STRONG", STRONG_C, "Good! Your model classifies this password as strong.", 3),
    }

    # Supports numeric labels 0, 1, 2 as used in the notebook.
    try:
        key = int(prediction)
    except (TypeError, ValueError):
        key = prediction

    return labels.get(
        key,
        ("UNKNOWN", UNKNOWN_C, f"Model returned class: {prediction}", 0)
    )


# --------------------------------- Theme ------------------------------------

BG_TOP = "#2A0A10"
BG_BOTTOM = "#060204"
GLOW = "#6B1D27"

CARD = "#150A0E"
CARD_BORDER = "#3A1A21"
FIELD = "#0C0508"
FIELD_BORDER = "#3A1A21"

TEXT = "#F8F4F5"
MUTED = "#9C8A8F"
ACCENT = "#F0636F"
ACCENT_DARK = "#C8454F"
ACCENT_SOFT = "#3A1319"
DIM = "#2A1419"

FONT = "Segoe UI"
MONO = "Consolas"


def hex_to_rgb(h):
    h = h.lstrip("#")
    return tuple(int(h[i:i + 2], 16) for i in (0, 2, 4))


def blend(c1, c2, t):
    a, b = hex_to_rgb(c1), hex_to_rgb(c2)
    return "#%02x%02x%02x" % tuple(int(a[i] + (b[i] - a[i]) * t) for i in range(3))


def add_hover(widget, normal_bg, hover_bg, normal_fg=None, hover_fg=None):
    def on_enter(_e):
        widget.configure(bg=hover_bg)
        if hover_fg:
            widget.configure(fg=hover_fg)

    def on_leave(_e):
        widget.configure(bg=normal_bg)
        if normal_fg:
            widget.configure(fg=normal_fg)

    widget.bind("<Enter>", on_enter)
    widget.bind("<Leave>", on_leave)


# --------------------------------- UI ---------------------------------------

class PasswordStrengthApp:
    PAD = 28

    def __init__(self, root):
        self.root = root
        self.root.title("Password Strength Checker")
        self.root.geometry("640x620")
        self.root.minsize(560, 580)
        self.root.configure(bg=BG_BOTTOM)

        self.password_visible = False
        self.password_var = tk.StringVar()
        self._anim_job = None

        self.canvas = tk.Canvas(root, highlightthickness=0, bd=0, bg=BG_BOTTOM)
        self.canvas.pack(fill="both", expand=True)

        self.card = self._build_card()
        self.card_window = self.canvas.create_window(0, 0, window=self.card, anchor="n")

        self.canvas.bind("<Configure>", self._on_resize)
        self.password_var.trace_add("write", self._on_password_change)
        self.password_entry.focus_set()

    # ---------------------------- Background / header ----------------------------

    def _on_resize(self, event):
        w, h = event.width, event.height
        self._draw_background(w, h)
        self._draw_header(w)
        self.canvas.coords(self.card_window, w / 2, 128)
        self.canvas.itemconfigure(self.card_window, width=min(w - 2 * self.PAD, 600))

    def _draw_background(self, w, h):
        c = self.canvas
        c.delete("bg")
        step = 3
        for y in range(0, h, step):
            c.create_rectangle(0, y, w, y + step, fill=blend(BG_TOP, BG_BOTTOM, y / max(h, 1)),
                               outline="", tags="bg")

        # Soft radial glow, top-right (stepped ovals approximate a gradient).
        cx, cy, rmax, layers = w * 0.88, h * 0.12, max(w, h) * 0.55, 34
        for i in range(layers):
            t = i / (layers - 1)
            r = rmax * (1 - t)
            base = blend(BG_TOP, BG_BOTTOM, cy / max(h, 1))
            colour = blend(base, GLOW, (t ** 1.6) * 0.85)
            c.create_oval(cx - r, cy - r, cx + r, cy + r, fill=colour, outline="", tags="bg")

        c.tag_lower("bg")

    @staticmethod
    def _round_rect(canvas, x1, y1, x2, y2, r, **kw):
        pts = [x1 + r, y1, x2 - r, y1, x2, y1, x2, y1 + r, x2, y2 - r, x2, y2,
               x2 - r, y2, x1 + r, y2, x1, y2, x1, y2 - r, x1, y1 + r, x1, y1]
        return canvas.create_polygon(pts, smooth=True, **kw)

    def _draw_header(self, w):
        c = self.canvas
        c.delete("header")
        x = self.PAD

        # Logo badge
        self._round_rect(c, x, 20, x + 38, 58, 11, fill=ACCENT_SOFT, outline="#6B2630",
                         width=1, tags="header")
        c.create_text(x + 19, 39, text="◆", fill=ACCENT, font=(FONT, 14, "bold"), tags="header")

        # Brand + subtitle
        c.create_text(x + 50, 31, text="P A S S G U A R D", fill=ACCENT, anchor="w",
                      font=(FONT, 8, "bold"), tags="header")
        c.create_text(x + 50, 48, text="machine-learning strength analysis", fill=MUTED,
                      anchor="w", font=(FONT, 8), tags="header")

        # Title
        c.create_text(x, 84, text="Password Strength Checker", fill=TEXT, anchor="w",
                      font=(FONT, 18, "bold"), tags="header")
        c.create_line(x, 104, x + 48, 104, fill=ACCENT, width=2, tags="header")

        # Status pill (top-right)
        px2 = w - self.PAD
        self._round_rect(c, px2 - 104, 26, px2, 48, 11, fill="#1E0C11", outline="#4A1F27",
                         tags="header")
        c.create_oval(px2 - 94, 34, px2 - 88, 40, fill=STRONG_C, outline="", tags="header")
        c.create_text(px2 - 80, 37, text="Model ready", fill=TEXT, anchor="w",
                      font=(FONT, 8), tags="header")

    # ---------------------------------- Card ----------------------------------

    def _build_card(self):
        card = tk.Frame(self.canvas, bg=CARD, highlightbackground=CARD_BORDER,
                        highlightthickness=1)
        content = tk.Frame(card, bg=CARD)
        content.pack(fill="both", expand=True, padx=22, pady=18)

        # Label
        tk.Label(content, text="E N T E R   P A S S W O R D", bg=CARD, fg=ACCENT,
                 font=(FONT, 9, "bold")).pack(anchor="w", pady=(0, 10))

        # Input row
        self.input_row = tk.Frame(content, bg=FIELD, highlightbackground=FIELD_BORDER,
                                  highlightcolor=ACCENT, highlightthickness=1)
        self.input_row.pack(fill="x")

        self.password_entry = tk.Entry(
            self.input_row, textvariable=self.password_var, show="•",
            bg=FIELD, fg=TEXT, insertbackground=ACCENT, relief="flat", bd=0,
            font=(MONO, 12)
        )
        self.password_entry.pack(side="left", fill="x", expand=True, padx=(12, 6), ipady=9)
        self.password_entry.bind("<Return>", lambda _e: self.check_password())
        self.password_entry.bind("<FocusIn>", lambda _e: self.input_row.configure(
            highlightbackground=ACCENT))
        self.password_entry.bind("<FocusOut>", lambda _e: self.input_row.configure(
            highlightbackground=FIELD_BORDER))

        self.toggle_button = tk.Button(
            self.input_row, text="SHOW", command=self.toggle_visibility,
            bg=FIELD, fg=ACCENT, activebackground=FIELD, activeforeground=TEXT,
            relief="flat", bd=0, cursor="hand2", font=(FONT, 9, "bold")
        )
        self.toggle_button.pack(side="right", padx=16)
        add_hover(self.toggle_button, FIELD, FIELD, ACCENT, TEXT)

        # Character chips (live)
        chips = tk.Frame(content, bg=CARD)
        chips.pack(fill="x", pady=(10, 0))
        self.chips = {}
        for key, text in (("len", "0 chars"), ("lower", "a–z"), ("upper", "A–Z"),
                          ("digit", "0–9"), ("symbol", "!@#")):
            lbl = tk.Label(chips, text=text, bg=DIM, fg=MUTED, font=(FONT, 8),
                           padx=8, pady=3)
            lbl.pack(side="left", padx=(0, 6))
            self.chips[key] = lbl

        # Buttons
        actions = tk.Frame(content, bg=CARD)
        actions.pack(fill="x", pady=(14, 14))

        self.check_button = tk.Button(
            actions, text="Analyze password", command=self.check_password,
            bg=ACCENT, fg="#1A0508", activebackground=ACCENT_DARK, activeforeground="#1A0508",
            relief="flat", bd=0, cursor="hand2", font=(FONT, 10, "bold"), padx=20, pady=7
        )
        self.check_button.pack(side="left")
        add_hover(self.check_button, ACCENT, "#FF7A85")

        self.clear_button = tk.Button(
            actions, text="Clear", command=self.clear,
            bg=DIM, fg="#D9C9CD", activebackground="#3A1B22", activeforeground=TEXT,
            relief="flat", bd=0, cursor="hand2", font=(FONT, 10), padx=16, pady=7
        )
        self.clear_button.pack(side="left", padx=(12, 0))
        add_hover(self.clear_button, DIM, "#3A1B22")

        # Result panel
        result = tk.Frame(content, bg="#1B0C11", highlightbackground="#4A1F27",
                          highlightthickness=1)
        result.pack(fill="x")
        inner = tk.Frame(result, bg="#1B0C11")
        inner.pack(fill="x", padx=16, pady=12)

        tk.Label(inner, text="P R E D I C T I O N", bg="#1B0C11", fg=MUTED,
                 font=(FONT, 9, "bold")).pack(anchor="w")

        self.result_title = tk.Label(inner, text="Awaiting input", bg="#1B0C11", fg=TEXT,
                                     font=(FONT, 18, "bold"))
        self.result_title.pack(anchor="w", pady=(4, 2))

        self.result_message = tk.Label(
            inner, text="Type a password and press Analyze to see the model's verdict.",
            bg="#1B0C11", fg=MUTED, font=(FONT, 9), wraplength=460, justify="left"
        )
        self.result_message.pack(anchor="w", pady=(0, 10))

        # Segmented meter
        self.meter = tk.Canvas(inner, height=8, bg="#1B0C11", highlightthickness=0)
        self.meter.pack(fill="x")
        self.meter.bind("<Configure>", lambda _e: self._draw_meter(self._meter_level,
                                                                    self._meter_color))
        self._meter_level, self._meter_color = 0, DIM

        # Footer note
        tk.Label(
            content,
            text="Runs 100% locally  •  nothing is uploaded, stored, or logged",
            bg=CARD, fg="#6F5B61", font=(FONT, 9)
        ).pack(anchor="w", pady=(12, 0))

        return card

    # ---------------------------------- Meter ----------------------------------

    def _draw_meter(self, level, color):
        self._meter_level, self._meter_color = level, color
        m = self.meter
        m.delete("all")
        w = m.winfo_width()
        if w < 10:
            return
        gap, segs = 8, 3
        seg_w = (w - gap * (segs - 1)) / segs
        for i in range(segs):
            x1 = i * (seg_w + gap)
            m.create_rectangle(x1, 0, x1 + seg_w, 8,
                               fill=color if i < level else DIM, outline="")

    def _animate_meter(self, target, color, current=0):
        if self._anim_job:
            self.root.after_cancel(self._anim_job)
            self._anim_job = None
        self._draw_meter(current, color)
        if current < target:
            self._anim_job = self.root.after(
                140, lambda: self._animate_meter(target, color, current + 1))

    # ---------------------------------- Logic ----------------------------------

    def _on_password_change(self, *_):
        pw = self.password_var.get()
        flags = {
            "lower": any(c.islower() for c in pw),
            "upper": any(c.isupper() for c in pw),
            "digit": any(c.isdigit() for c in pw),
            "symbol": any(not c.isalnum() for c in pw),
        }
        self.chips["len"].configure(
            text=f"{len(pw)} chars", bg=ACCENT_SOFT if pw else DIM, fg=ACCENT if pw else MUTED)
        for key, on in flags.items():
            self.chips[key].configure(bg=ACCENT_SOFT if on else DIM, fg=ACCENT if on else MUTED)
        self._reset_result()

    def _reset_result(self):
        self.result_title.configure(text="Awaiting input", fg=TEXT)
        self.result_message.configure(
            text="Type a password and press Analyze to see the model's verdict.")
        self._animate_meter(0, DIM)

    def toggle_visibility(self):
        self.password_visible = not self.password_visible
        self.password_entry.configure(show="" if self.password_visible else "•")
        self.toggle_button.configure(text="HIDE" if self.password_visible else "SHOW")

    def check_password(self):
        password = self.password_var.get()

        if not password:
            messagebox.showwarning("Password required", "Please enter a password first.")
            self.password_entry.focus_set()
            return

        try:
            label, color, description, level = predict_strength(password)
        except Exception as error:
            messagebox.showerror("Prediction error", str(error))
            return

        self.result_title.configure(text=label.title(), fg=color)
        self.result_message.configure(text=description)
        self._animate_meter(level, color)

    def clear(self):
        self.password_var.set("")
        self.password_entry.focus_set()


if __name__ == "__main__":
    try:
        model, vectorizer = load_artifacts()
    except Exception as error:
        startup = tk.Tk()
        startup.withdraw()
        messagebox.showerror("Could not load model files", str(error))
        startup.destroy()
        raise SystemExit(1)

    root = tk.Tk()
    app = PasswordStrengthApp(root)
    root.mainloop()

C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
